# Trade the 8-K: research notebook

**Gator Quant Hacks 2026 · Massive Challenge**

This notebook is built on the organizers' starter notebook, with these changes:

1. **Several tags as one event.** `EVENT_TAGS` takes a list of tags, collapsed to one event per filer per filing date.
2. **Timing fixed before pricing.** EDGAR acceptance times are applied inside `build_events`, so filings accepted after 4:00 PM ET enter on the next session. The sealed-window run gets the same fix.
3. **Optional earnings filter.** Events within `EXCLUDE_EARNINGS_DAYS` of an earnings-related 8-K for the same company are dropped.
4. **Strangle and switch legs.** The scoreboard also reports a short strangle (benchmark) and the two "switch values": buying back the short call (`drop_call`) or the short put (`drop_put`) at entry.
5. **One function for every window.** In-sample, out-of-sample and the judges' sealed window all call the same `run_study(start, end)`, so the sealed run uses exactly your rule.
6. **Variant log.** Every run appends a line to `outputs/variants_log.csv`, so you can report how many variants you tried.

**How to use it**

1. Fill in the hypothesis cell below **before** looking at results, and commit it to git so the timestamp is on record.
2. Set the configuration cell (section 2) and run all cells.
3. To try another idea, change the configuration and re-run. Every run is logged.
4. Fill in the write-up at the end.

## 0 · Hypothesis (write before running)

*Write this and commit it before running sections 4 onwards. Edit only by adding dated notes underneath, never by rewriting.*

- **Date and commit:**
- **Categories (tags):**
- **Strategy (one of the five):**
- **Thesis shape:** moves more than priced and direction is callable / overpriced, drama does not arrive / must hold through the event
- **Economic reason the market should misprice this category:**
- **Rule:** which filings, which contracts (bucket, OTM), entry (`post` at the close of the corrected filing session), exit (horizon)
- **Predicted sign at each horizon, compared with ordinary days:**
- **What would make it break (predicted fragility):**

*Dated notes (append only):*

## 1 · Setup

Needs Python 3.10+, `pandas numpy requests matplotlib`, and `MASSIVE_API_KEY` in the environment or in a `.env` file next to this notebook. API responses are cached in `.massive_cache/`.

In [1]:
import os, json, time, hashlib, re, math
from dataclasses import dataclass
from pathlib import Path
from getpass import getpass
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday, USMartinLutherKingJr,
                                    USPresidentsDay, GoodFriday, USMemorialDay, USLaborDay, USThanksgivingDay)
from pandas.tseries.offsets import CustomBusinessDay

BASE_URL = "https://api.massive.com"
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)


def load_api_key(name: str = "MASSIVE_API_KEY") -> str:
    key = (os.environ.get(name) or "").strip()
    env_file = Path(".env")
    if not key and env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith(f"{name}="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not key:
        key = getpass(f"{name}: ").strip()
    return key


API_KEY = load_api_key()
SESSION = requests.Session()
SESSION.headers["Authorization"] = f"Bearer {API_KEY}"
CACHE_DIR = Path(".massive_cache"); CACHE_DIR.mkdir(exist_ok=True)


def api_get(path_or_url: str, params: dict | None = None) -> dict:
    url = path_or_url if path_or_url.startswith("http") else BASE_URL + path_or_url
    full_url = requests.Request("GET", url, params=params).prepare().url
    cache_file = CACHE_DIR / (hashlib.sha1(full_url.encode()).hexdigest() + ".json")
    if cache_file.exists():
        return json.loads(cache_file.read_text())
    for attempt in range(10):
        resp = SESSION.get(full_url, timeout=60)
        if resp.status_code in (429, 500, 502, 503, 504):
            ra = resp.headers.get("Retry-After", "")
            time.sleep(float(ra) if ra.isdigit() else min(2 ** attempt, 20))
            continue
        break
    resp.raise_for_status()
    payload = resp.json()
    cache_file.write_text(json.dumps(payload))
    return payload


def api_get_all(path: str, params: dict | None = None, max_pages: int = 500) -> list[dict]:
    payload = api_get(path, params)
    rows = list(payload.get("results") or [])
    pages = 1
    while payload.get("next_url") and pages < max_pages:
        payload = api_get(payload["next_url"])
        rows.extend(payload.get("results") or [])
        pages += 1
    return rows


print(f"API key loaded (ends {API_KEY[-4:]})")

API key loaded (ends 0itp)


In [24]:
# ── Robust api_get: also retry on timeouts / dropped connections ──
from requests.exceptions import Timeout, ConnectionError as ReqConnError, ChunkedEncodingError

def api_get(path_or_url: str, params: dict | None = None) -> dict:
    url = path_or_url if path_or_url.startswith("http") else BASE_URL + path_or_url
    full_url = requests.Request("GET", url, params=params).prepare().url
    cache_file = CACHE_DIR / (hashlib.sha1(full_url.encode()).hexdigest() + ".json")
    if cache_file.exists():
        return json.loads(cache_file.read_text())
    resp = None
    for attempt in range(8):
        try:
            resp = SESSION.get(full_url, timeout=(10, 45))
        except (Timeout, ReqConnError, ChunkedEncodingError) as err:
            wait = min(2 ** attempt, 30)
            print(f"   network error ({type(err).__name__}), retry {attempt + 1}/8 in {wait}s")
            time.sleep(wait); continue
        if resp.status_code in (429, 500, 502, 503, 504):
            ra = resp.headers.get("Retry-After", "")
            time.sleep(float(ra) if ra.isdigit() else min(2 ** attempt, 20)); continue
        break
    if resp is None:
        raise RuntimeError(f"gave up after 8 network errors: {full_url[:120]}")
    resp.raise_for_status()
    payload = resp.json()
    cache_file.write_text(json.dumps(payload))
    return payload

## 2 · Configuration

Everything you change lives here. `HORIZONS` is fixed by the organizers; do not change it.

- `EVENT_TAGS`: one or more `tertiary_category` tags, treated as one event type.
- `HEADLINE_STRATEGY`: the strategy you registered in section 0. It is used for the trade specification and the variant log, so the "winner" is your pre-registered choice, not whichever looks best.
- Switch columns: `drop_call` is the value of buying back the short call at entry (switching a strangle to a cash-secured put); `drop_put` is the value of buying back the short put (switching toward a covered call or collar).

In [ ]:
# ---- Event definition ---------------------------------------------------------------------------
EVENT_TAGS        = ["ceo_departure", "cfo_departure"]
HEADLINE_STRATEGY = "protective_put"
ENTRY             = "pre"          # pricing question; section 11 also shows post
VARIANT_NOTE      = "B gov shock; pre-entry pricing test; post = tradability"
RUN_PLACEBO       = True
EXCLUDE_EARNINGS_DAYS = 1          # make sure departures announced with earnings aren't driving it

# ---- Windows ------------------------------------------------------------------------------------
STUDY_START, STUDY_END = "2024-01-01", "2025-12-31"     # in-sample
OOS_START, OOS_END = "2026-01-01", "2026-08-31"         # out-of-sample (do not tune on it)
HOLDOUT_START, HOLDOUT_END = "2023-06-01", "2023-08-31" # judges replace these
RUN_HOLDOUT = False                                     # judges flip this

# ---- Universe (static top 100, as in the starter) ----------------------------------------------
TOP_100 = """
AAPL ABBV ABT ACN ADBE AIG AMD AMGN AMT AMZN AVGO AXP BA BAC BK BKNG BLK BMY BRK.B C
CAT CHTR CL CMCSA COF COP COST CRM CSCO CVS CVX DE DHR DIS DUK EMR FDX GD GE GILD
GM GOOGL GS HD HON IBM INTC INTU ISRG JNJ JPM KO LIN LLY LMT LOW MA MCD MDLZ MDT
MET META MMM MO MRK MS MSFT NEE NFLX NKE NOW NVDA ORCL PEP PFE PG PLTR PM PYPL QCOM
RTX SBUX SCHW SO T TGT TMO TMUS TSLA TXN UBER UNH UNP UPS USB V VZ WFC WMT XOM
""".split()
assert len(TOP_100) == 100 and len(set(TOP_100)) == 100

# ---- Options -------------------------------------------------------------------------------------
EXPIRY_BUCKETS = {"1m": (21, 45, 30), "2m": (46, 80, 60), "3-6m": (90, 180, 120)}
BASELINE_BUCKET = "3-6m"
HORIZONS = [1, 2, 3, 5, 10, 21, 42, 63]   # fixed for every team
OTM_PCT = 0.05
OTM_GRID = [0.03, 0.05, 0.10]
ENTRY = "post"                            # "post" is tradeable; "pre" asks whether the market priced it
RISK_FREE = 0.04
STRIKE_WINDOW = 0.25
MAX_STALE_SESSIONS = 3
MAX_EVENTS = None                         # e.g. 15 for a quick smoke test

# ---- Filters and timing --------------------------------------------------------------------------
FETCH_ACCEPTANCE_TIMES = True             # move after-close filings to the next session (needs sec.gov)
SEC_USER_AGENT = "GatorQuantHacks team-name your@email.edu"   # put a real contact here
EXCLUDE_EARNINGS_DAYS = 0                 # e.g. 3 drops events within 3 days of an earnings-related 8-K

# ---- Placebo, costs, extras ----------------------------------------------------------------------
RUN_PLACEBO = True
N_PLACEBO = 120
PLACEBO_GAP_DAYS = 30
COST_HAIRCUT = 0.05                       # fraction of each traded premium, per side
RUN_TAG_SCAN = False                      # count events per tag in the universe (about 119 paginated requests)
SENS_HORIZON = 21

assert OTM_PCT in OTM_GRID and BASELINE_BUCKET in EXPIRY_BUCKETS and ENTRY in ("pre", "post")
for _n in ("STUDY_START", "STUDY_END", "OOS_START", "OOS_END", "HOLDOUT_START", "HOLDOUT_END"):
    pd.Timestamp(globals()[_n])

## 3 · Taxonomy, calendar and tag scan

The taxonomy cell checks that every tag in `EVENT_TAGS` exists, and finds the earnings-related tags used by the earnings filter. The optional tag scan (`RUN_TAG_SCAN = True`) counts in-sample events per tag in the universe, so you only build hypotheses on tags with enough events.

In [6]:
taxonomy = pd.DataFrame(api_get_all("/stocks/taxonomies/vX/disclosures", {"limit": 1000}))
ALL_TAGS = set(taxonomy.tertiary_category)
_bad = [t for t in EVENT_TAGS if t not in ALL_TAGS]
assert not _bad, f"Unknown tags: {_bad}"

_txt = (taxonomy.primary_category + " " + taxonomy.secondary_category + " " + taxonomy.tertiary_category).str.lower()
EARNINGS_TAGS = sorted(taxonomy.loc[_txt.str.contains("earning|financial_result"), "tertiary_category"].unique())
print(f"{len(taxonomy)} tags. Earnings-related tags used by the filter: {EARNINGS_TAGS}")

with pd.option_context("display.max_colwidth", 90):
    display(taxonomy[taxonomy.tertiary_category.isin(EVENT_TAGS)][["secondary_category", "tertiary_category", "description"]])


class NYSEHolidays(AbstractHolidayCalendar):
    rules = [
        Holiday("New Year's Day", month=1, day=1, observance=lambda d: d + pd.Timedelta(days=1) if d.weekday() == 6 else d),
        USMartinLutherKingJr, USPresidentsDay, GoodFriday, USMemorialDay,
        Holiday("Juneteenth", month=6, day=19, start_date="2022-01-01", observance=nearest_workday),
        Holiday("Independence Day", month=7, day=4, observance=nearest_workday),
        USLaborDay, USThanksgivingDay,
        Holiday("Christmas Day", month=12, day=25, observance=nearest_workday),
        Holiday("National day of mourning, President Carter", year=2025, month=1, day=9),
    ]


def trading_sessions(start, end):
    hol = NYSEHolidays().holidays(pd.Timestamp(start) - pd.Timedelta(days=7), pd.Timestamp(end) + pd.Timedelta(days=7))
    return pd.bdate_range(start, end, freq=CustomBusinessDay(holidays=hol))


CAL = trading_sessions("2021-06-01", "2027-12-31")
TODAY = pd.Timestamp.today().normalize()
LAST_SESSION = CAL[CAL.searchsorted(TODAY, side="right") - 1]
session_on_or_after = lambda d: CAL[CAL.searchsorted(pd.Timestamp(d), side="left")]
session_before = lambda d: CAL[CAL.searchsorted(pd.Timestamp(d), side="left") - 1]
next_session = lambda d: CAL[CAL.searchsorted(pd.Timestamp(d), side="right")]


def sessions_between(a, b) -> int:
    return int(CAL.searchsorted(pd.Timestamp(b), side="right") - CAL.searchsorted(pd.Timestamp(a), side="right"))

119 tags. Earnings-related tags used by the filter: ['accounting_error_correction', 'annual_earnings', 'asset_impairment', 'audit_opinion_withdrawal', 'financial_restatement', 'goodwill_impairment', 'guidance_issuance_or_update', 'guidance_withdrawal', 'internal_control_weakness', 'investment_impairment', 'material_charge_or_gain', 'nav_per_share', 'preliminary_results', 'quarterly_earnings']


,secondary_category,tertiary_category,description
2,executive_leadership,cfo_appointment,"New CFO appointment with background, terms, and transition timeline."


In [8]:
def normalize_ticker(t):
    if not isinstance(t, str) or not t.strip():
        return None
    return t.strip().upper().replace("/", ".")


def fetch_disclosures(tag, start, end):
    rows = api_get_all("/stocks/filings/8-K/vX/disclosures", {
        "tertiary_category": tag, "filing_date.gte": start, "filing_date.lte": end,
        "limit": 1000, "sort": "filing_date.asc"})
    df = pd.DataFrame(rows)
    if not df.empty:
        df["filing_date"] = pd.to_datetime(df["filing_date"])
        df["tag"] = tag
        if "tickers" not in df.columns:
            df["tickers"] = df["ticker"] if "ticker" in df.columns else None
        df["tickers"] = df["tickers"].map(
            lambda t: t if isinstance(t, list) else ([t] if isinstance(t, str) and t else []))
        for c in ("cik", "filing_url", "supporting_text", "accession_number"):
            if c not in df.columns:
                df[c] = None
    return df


if RUN_TAG_SCAN:
    counts = []
    for i, tag in enumerate(sorted(ALL_TAGS), 1):
        raw = fetch_disclosures(tag, STUDY_START, STUDY_END)
        if raw.empty:
            counts.append((tag, 0, 0)); continue
        ex = raw.explode("tickers"); ex["ticker"] = ex["tickers"].map(normalize_ticker)
        ex = ex[ex.ticker.isin(TOP_100)].drop_duplicates(["cik", "filing_date"])
        counts.append((tag, len(ex), ex.ticker.nunique()))
        if i % 20 == 0:
            print(f"  scanned {i}/{len(ALL_TAGS)} tags")
    tag_counts = (pd.DataFrame(counts, columns=["tag", "events_in_top100", "tickers"])
                    .merge(taxonomy[["tertiary_category", "secondary_category"]], left_on="tag", right_on="tertiary_category")
                    .drop(columns="tertiary_category").sort_values("events_in_top100", ascending=False))
    tag_counts.to_csv(OUT_DIR / "tag_counts_in_sample.csv", index=False)
    display(tag_counts.head(40))
else:
    print("Tag scan skipped (set RUN_TAG_SCAN = True).")

  scanned 20/119 tags
  scanned 40/119 tags
  scanned 60/119 tags
  scanned 80/119 tags
  scanned 100/119 tags


,tag,events_in_top100,tickers,secondary_category
40,debt_issuance,277,80,debt_activity
52,executive_compensation_change,174,79,executive_leadership
8,annual_meeting_results,150,87,shareholder_meetings
66,investor_presentation,149,39,investor_communications
114,underwriting_agreement,146,67,equity_activity
54,executive_officer_departure,132,68,executive_leadership
93,quarterly_earnings,131,27,earnings_and_performance
104,shareholder_proposal_outcome,121,76,shareholder_activism
53,executive_officer_appointment,105,62,executive_leadership
44,director_departure,101,59,board_of_directors


## 4 · Events (several tags, timing fix, earnings filter)

- `t_0` is the first session on or after the filing date, moved to the next session when EDGAR accepted the filing at or after 4:00 PM ET.
- `t_pre` is the session before `t_0`; the chain there cannot know the news.
- One event per filer per filing date, whichever of your tags it carries (the `tags` column lists them).

In [11]:
def fetch_acceptance_time(filing_url: str):
    if not isinstance(filing_url, str) or not filing_url:
        return None
    key = CACHE_DIR / ("sec_" + hashlib.sha1(filing_url.encode()).hexdigest() + ".txt")
    try:
        if key.exists():
            head = key.read_text()
        else:
            resp = requests.get(filing_url, headers={"User-Agent": SEC_USER_AGENT}, timeout=30, stream=True)
            resp.raise_for_status()
            chunk = next(resp.iter_content(4096))
            head = chunk.decode("utf-8", "ignore") if isinstance(chunk, bytes) else chunk
            key.write_text(head)
            time.sleep(0.12)
    except Exception:
        return None
    m = re.search(r"<ACCEPTANCE-DATETIME>(\d{14})", head)
    return pd.Timestamp(m.group(1)) if m else None


def earnings_dates(start: str, end: str, universe: list[str]) -> dict:
    if not EARNINGS_TAGS or EXCLUDE_EARNINGS_DAYS <= 0:
        return {}
    pad = pd.Timedelta(days=EXCLUDE_EARNINGS_DAYS + 7)
    s, e = (pd.Timestamp(start) - pad).strftime("%Y-%m-%d"), (pd.Timestamp(end) + pad).strftime("%Y-%m-%d")
    frames = [f for f in (fetch_disclosures(t, s, e) for t in EARNINGS_TAGS) if not f.empty]
    if not frames:
        return {}
    ex = pd.concat(frames).explode("tickers")
    ex["ticker"] = ex["tickers"].map(normalize_ticker)
    ex = ex[ex.ticker.isin(universe)]
    return ex.groupby("ticker")["filing_date"].apply(list).to_dict()


def build_events(tags: list[str], start: str, end: str, universe: list[str]) -> pd.DataFrame:
    frames = [f for f in (fetch_disclosures(t, start, end) for t in tags) if not f.empty]
    cols = ["ticker", "filing_date", "event_date", "t_pre", "t_0", "tags"]
    if not frames:
        print(f"No disclosures for {tags} in {start}..{end}")
        return pd.DataFrame(columns=cols)
    raw = pd.concat(frames, ignore_index=True)
    ex = raw.explode("tickers").rename(columns={"tickers": "ticker"})
    ex["ticker"] = ex["ticker"].map(normalize_ticker)
    ex = ex[ex["ticker"].isin(universe)]
    if ex.empty:
        print(f"No events in the universe for {tags} in {start}..{end}")
        return pd.DataFrame(columns=cols)
    ev = (ex.sort_values(["cik", "filing_date"])
            .groupby(["cik", "filing_date"], as_index=False)
            .agg(ticker=("ticker", "first"), filing_url=("filing_url", "first"),
                 supporting_text=("supporting_text", "first"),
                 tags=("tag", lambda s: ",".join(sorted(set(s)))))
            .sort_values("filing_date").reset_index(drop=True))
    ev["event_date"] = ev["filing_date"]
    ev["t_0"] = ev["filing_date"].map(session_on_or_after)
    if FETCH_ACCEPTANCE_TIMES:
        ev["accepted_at"] = pd.to_datetime(pd.Series([fetch_acceptance_time(u) for u in ev["filing_url"]], index=ev.index))
        late = ev["accepted_at"].dt.hour.ge(16).fillna(False).astype(bool)
        ev.loc[late, "t_0"] = ev.loc[late, "t_0"].map(next_session)
        ev["after_close"] = late
        print(f"  acceptance times: {ev.accepted_at.notna().mean():.0%} found, {late.mean():.0%} after the close (moved a session)")
    ev["t_pre"] = ev["t_0"].map(session_before)
    ed = earnings_dates(start, end, universe)
    if ed:
        near = [any(abs((d - x).days) <= EXCLUDE_EARNINGS_DAYS for x in ed.get(t, [])) for t, d in zip(ev.ticker, ev.filing_date)]
        print(f"  earnings filter: dropped {sum(near)} of {len(ev)} events within {EXCLUDE_EARNINGS_DAYS} days of an earnings 8-K")
        ev = ev[~np.array(near)].reset_index(drop=True)
    print(f"{len(ev)} events across {ev.ticker.nunique()} tickers for {tags}, {start}..{end}")
    return ev

## 5 · Option chain, recovered spot and leg pricing

Unchanged from the starter: spot comes from put-call parity at the money, then the expiry per bucket, then the strikes, then each leg's full bar history.

In [12]:
def option_bars(opt_ticker: str, start, end) -> pd.DataFrame:
    rows = api_get_all(f"/v2/aggs/ticker/{opt_ticker}/range/1/day/{pd.Timestamp(start):%Y-%m-%d}/{pd.Timestamp(end):%Y-%m-%d}",
                       {"adjusted": "false", "sort": "asc", "limit": 50000})
    if not rows:
        return pd.DataFrame(columns=["close", "volume"], index=pd.DatetimeIndex([], name="session"))
    idx = pd.to_datetime([r["t"] for r in rows], unit="ms", utc=True).tz_convert("America/New_York").normalize().tz_localize(None)
    return pd.DataFrame({"close": [float(r["c"]) for r in rows], "volume": [float(r.get("v") or 0) for r in rows]},
                        index=pd.DatetimeIndex(idx, name="session"))


def fetch_chain(ticker, as_of, dte_lo, dte_hi) -> pd.DataFrame:
    rows = api_get_all("/v3/reference/options/contracts", {
        "underlying_ticker": ticker, "as_of": as_of.strftime("%Y-%m-%d"),
        "expiration_date.gte": (as_of + pd.Timedelta(days=dte_lo)).strftime("%Y-%m-%d"),
        "expiration_date.lte": (as_of + pd.Timedelta(days=dte_hi)).strftime("%Y-%m-%d"), "limit": 1000})
    chain = pd.DataFrame(rows)
    if chain.empty:
        return chain
    if "shares_per_contract" in chain:
        chain = chain[chain["shares_per_contract"].fillna(100) == 100]
    chain = chain[["ticker", "contract_type", "strike_price", "expiration_date"]].copy()
    chain["expiration_date"] = pd.to_datetime(chain["expiration_date"])
    chain["dte"] = (chain["expiration_date"] - as_of).dt.days
    chain["strike_price"] = chain["strike_price"].astype(float)
    return chain.reset_index(drop=True)


def paired_strikes(e):
    both = e.groupby("strike_price")["contract_type"].nunique()
    return both[both == 2].index.to_numpy(dtype=float)


def contract(e, strike, kind):
    return e[(e.strike_price == strike) & (e.contract_type == kind)]["ticker"].iloc[0]


def last_close_on_or_before(opt_ticker, day, lookback_days=7):
    bars = option_bars(opt_ticker, day - pd.Timedelta(days=lookback_days), day)
    return float(bars["close"].iloc[-1]) if len(bars) else None


def locate_spot(chain, day, max_iter=8):
    near = chain[chain.dte >= 3]
    if near.empty:
        return None
    near = near[near.dte == near.dte.min()]
    strikes = paired_strikes(near)
    if len(strikes) < 3:
        return None
    T = near.dte.iloc[0] / 365
    k, tried, est = strikes[np.abs(strikes - np.median(strikes)).argmin()], set(), None
    for _ in range(max_iter):
        tried.add(k)
        c = last_close_on_or_before(contract(near, k, "call"), day)
        p = last_close_on_or_before(contract(near, k, "put"), day)
        if c is None or p is None:
            rest = [s for s in strikes if s not in tried]
            if not rest:
                break
            k = rest[int(np.abs(np.array(rest) - k).argmin())]
            continue
        est = k * np.exp(-RISK_FREE * T) + c - p
        k_new = strikes[np.abs(strikes - est).argmin()]
        if k_new == k or k_new in tried:
            break
        k = k_new
    return None if est is None else {"spot": float(est)}


def pick_expiry(chain, lo, hi, target):
    cand = chain[(chain.dte >= lo) & (chain.dte <= hi)]
    if cand.empty:
        return None
    dte_of = cand.groupby("expiration_date")["dte"].first()
    ok = [x for x, g in cand.groupby("expiration_date") if len(paired_strikes(g)) >= 3]
    return min(ok, key=lambda x: abs(dte_of[x] - target)) if ok else None


def select_strikes(e, spot, otm_pcts):
    both = paired_strikes(e)
    both = both[(both >= spot * (1 - STRIKE_WINDOW)) & (both <= spot * (1 + STRIKE_WINDOW))]
    if len(both) == 0:
        return None
    calls = np.sort(e.loc[e.contract_type == "call", "strike_price"].unique())
    puts = np.sort(e.loc[e.contract_type == "put", "strike_price"].unique())
    out = {"K": float(both[np.abs(both - spot).argmin()])}
    for pct in otm_pcts:
        up, dn = calls[calls >= spot * (1 + pct)], puts[puts <= spot * (1 - pct)]
        out[f"U{pct}"] = float(up.min()) if len(up) else float(calls.max())
        out[f"L{pct}"] = float(dn.max()) if len(dn) else float(puts.min())
    return out


@dataclass
class Leg:
    ticker: str
    kind: str
    strike: float
    bars: pd.DataFrame

    def mark(self, day):
        b = self.bars.loc[: pd.Timestamp(day)]
        if b.empty or sessions_between(b.index[-1], day) > MAX_STALE_SESSIONS:
            return np.nan
        return float(b["close"].iloc[-1])

    def volume_on(self, day):
        return float(self.bars["volume"].get(pd.Timestamp(day), 0.0))


@dataclass
class PricedEvent:
    ticker: str
    event_date: pd.Timestamp
    t_pre: pd.Timestamp
    t_0: pd.Timestamp
    bucket: str
    expiry: pd.Timestamp
    expiry_session: pd.Timestamp
    spot_pre: float
    strikes: dict
    legs: dict

    def marks(self, day):
        return {n: leg.mark(day) for n, leg in self.legs.items()}

    def synthetic_spot(self, day, m=None):
        m = m if m is not None else self.marks(day)
        T = max((self.expiry - pd.Timestamp(day)).days, 0) / 365
        return self.strikes["K"] * np.exp(-RISK_FREE * T) + m["C_K"] - m["P_K"]


def price_event(ticker, t_pre, t_0, event_date, buckets, otm_pcts):
    chain = fetch_chain(ticker, t_pre, 2, max(b[1] for b in buckets.values()))
    if chain.empty:
        return [], ["no option chain"]
    loc = locate_spot(chain, t_pre)
    if loc is None:
        return [], ["could not recover spot"]
    priced, notes = [], []
    for name, (lo, hi, target) in buckets.items():
        expiry = pick_expiry(chain, lo, hi, target)
        if expiry is None:
            notes.append(f"{name}: no expiry"); continue
        e = chain[chain.expiration_date == expiry]
        strikes = select_strikes(e, loc["spot"], otm_pcts)
        if strikes is None:
            notes.append(f"{name}: no strikes"); continue
        wanted = {"C_K": ("call", strikes["K"]), "P_K": ("put", strikes["K"])}
        for pct in otm_pcts:
            wanted[f"C_U{pct}"] = ("call", strikes[f"U{pct}"])
            wanted[f"P_L{pct}"] = ("put", strikes[f"L{pct}"])
        legs = {k: Leg(contract(e, s, kind), kind, s, option_bars(contract(e, s, kind), t_pre - pd.Timedelta(days=10), expiry))
                for k, (kind, s) in wanted.items()}
        pe = PricedEvent(ticker, event_date, t_pre, t_0, name, expiry, CAL[CAL.searchsorted(expiry, side="right") - 1],
                         loc["spot"], strikes, legs)
        if np.isnan(pe.legs["C_K"].mark(t_pre)) or np.isnan(pe.legs["P_K"].mark(t_pre)):
            notes.append(f"{name}: ATM pair did not trade near t_pre"); continue
        priced.append(pe)
    return priced, notes


def price_events(ev, buckets=None, otm_pcts=None, label="events"):
    buckets, otm_pcts = buckets or EXPIRY_BUCKETS, otm_pcts or OTM_GRID
    priced, dropped, t0 = [], [], time.time()
    for n, row in enumerate(ev.itertuples(index=False), 1):
        got, notes = price_event(row.ticker, row.t_pre, row.t_0, row.event_date, buckets, otm_pcts)
        priced += got
        dropped += [(row.ticker, row.t_0, x) for x in notes]
        if n % 25 == 0:
            print(f"  {label}: {n}/{len(ev)} events, {time.time() - t0:.0f}s")
    print(f"{label}: {len(ev)} events -> {len(priced)} priced (event, bucket) pairs; {len(dropped)} drops")
    return priced, pd.DataFrame(dropped, columns=["ticker", "t_0", "reason"])

## 6 · P&L engine: the five strategies, the strangle and the switch legs

All values are P&L per $1 of stock at entry. `S` is the synthetic stock from the at-the-money pair.

| Column | P&L | Meaning |
|---|---|---|
| `stock` | ΔS / S | Reference: the stock alone |
| `long_call` | ΔC_K / S | 1 · Long call |
| `covered_call` | (ΔS − ΔC_U) / S | 2 · Covered call |
| `protective_put` | (ΔS + ΔP_L) / S | 3 · Protective put |
| `collar` | (ΔS + ΔP_L − ΔC_U) / S | 4 · Collar |
| `cash_secured_put` | −ΔP_L / S | 5 · Cash-secured put |
| `short_strangle` | −(ΔC_U + ΔP_L) / S | Benchmark you switch out of (not a library strategy) |
| `drop_call` | ΔC_U / S | Switch value: buy back the short call (strangle → cash-secured put) |
| `drop_put` | ΔP_L / S | Switch value: buy back the short put (strangle → short call side) |

Note that `short_strangle + drop_call = cash_secured_put`. A positive event-minus-placebo `drop_call` means switching to the cash-secured put after this category beats switching on ordinary days.

In [13]:
STRATEGIES = ["stock", "long_call", "covered_call", "protective_put", "collar", "cash_secured_put",
              "short_strangle", "drop_call", "drop_put"]
STRATEGY_LABEL = {"stock": "Stock only (synthetic)", "long_call": "1 · Long call", "covered_call": "2 · Covered call",
                  "protective_put": "3 · Protective put", "collar": "4 · Collar", "cash_secured_put": "5 · Cash-secured put",
                  "short_strangle": "Short strangle (benchmark)", "drop_call": "Switch: buy back call",
                  "drop_put": "Switch: buy back put"}
LEGS_USED = {"long_call": ["C_K"], "covered_call": ["C_U"], "protective_put": ["P_L"], "collar": ["C_U", "P_L"],
             "cash_secured_put": ["P_L"], "short_strangle": ["C_U", "P_L"], "drop_call": ["C_U"], "drop_put": ["P_L"], "stock": []}
assert HEADLINE_STRATEGY in STRATEGIES


def strategy_pnl(m_e, m_x, S_e, S_x, otm):
    dS = S_x - S_e
    dC_U = m_x[f"C_U{otm}"] - m_e[f"C_U{otm}"]
    dP_L = m_x[f"P_L{otm}"] - m_e[f"P_L{otm}"]
    dC_K = m_x["C_K"] - m_e["C_K"]
    return {"stock": dS / S_e, "long_call": dC_K / S_e, "covered_call": (dS - dC_U) / S_e,
            "protective_put": (dS + dP_L) / S_e, "collar": (dS + dP_L - dC_U) / S_e,
            "cash_secured_put": -dP_L / S_e, "short_strangle": -(dC_U + dP_L) / S_e,
            "drop_call": dC_U / S_e, "drop_put": dP_L / S_e}


def evaluate(priced, otm_pcts=None):
    otm_pcts = otm_pcts or OTM_GRID
    rows = []
    for pe in priced:
        i0 = CAL.get_loc(pe.t_0)
        exits = {0: pe.t_0}
        exits.update({h: CAL[i0 + h] for h in HORIZONS if i0 + h < len(CAL) and CAL[i0 + h] <= pe.expiry_session})
        exits["exp"] = pe.expiry_session
        for entry, e_day in (("pre", pe.t_pre), ("post", pe.t_0)):
            m_e = pe.marks(e_day)
            S_e = pe.synthetic_spot(e_day, m_e)
            if np.isnan(S_e):
                continue
            implied = (m_e["C_K"] + m_e["P_K"]) / S_e
            dte = sessions_between(e_day, pe.expiry_session)
            for h, x_day in exits.items():
                if x_day > LAST_SESSION:
                    continue
                m_x = pe.marks(x_day)
                S_x = pe.synthetic_spot(x_day, m_x)
                held = sessions_between(e_day, x_day)
                base = {"ticker": pe.ticker, "event_date": pe.event_date, "t_0": pe.t_0, "bucket": pe.bucket,
                        "entry": entry, "horizon": h, "S_entry": S_e, "realized": S_x / S_e - 1, "implied_move": implied,
                        "implied_scaled": implied * np.sqrt(held / dte) if dte else np.nan}
                for otm in otm_pcts:
                    rows.append(dict(base, otm=otm, **strategy_pnl(m_e, m_x, S_e, S_x, otm)))
    res = pd.DataFrame(rows)
    if not res.empty:
        res["ratio"] = res["realized"].abs() / res["implied_scaled"]
    return res

In [17]:
# ── Extended strategies (supporting evidence; the headline stays a library strategy) ──
_EXT = {
    "long_put":          ("Ext · Long put (ATM)",          ["P_K"]),
    "long_straddle":     ("Ext · Long straddle (ATM)",     ["C_K", "P_K"]),
    "short_straddle":    ("Ext · Short straddle (ATM)",    ["C_K", "P_K"]),
    "short_iron_fly":    ("Ext · Short iron butterfly",    ["C_K", "P_K", "C_U", "P_L"]),
    "bull_call_spread":  ("Ext · Bull call spread",        ["C_K", "C_U"]),
    "bear_put_spread":   ("Ext · Bear put spread",         ["P_K", "P_L"]),
    "risk_reversal":     ("Ext · Risk reversal (+C_U −P_L)", ["C_U", "P_L"]),
    "atm_put_write":     ("Ext · ATM put-write",           ["P_K"]),
}
for k, (label, legs) in _EXT.items():
    if k not in STRATEGIES:
        STRATEGIES.append(k)          # in place, so default args in helpers see it
    STRATEGY_LABEL[k] = label
    LEGS_USED[k] = legs

_base_pnl = strategy_pnl.__wrapped__ if hasattr(strategy_pnl, "__wrapped__") else strategy_pnl

def strategy_pnl(m_e, m_x, S_e, S_x, otm):
    out = _base_pnl(m_e, m_x, S_e, S_x, otm)
    dC_K = m_x["C_K"] - m_e["C_K"]
    dP_K = m_x["P_K"] - m_e["P_K"]
    dC_U = m_x[f"C_U{otm}"] - m_e[f"C_U{otm}"]
    dP_L = m_x[f"P_L{otm}"] - m_e[f"P_L{otm}"]
    out.update({
        "long_put":         dP_K / S_e,
        "long_straddle":    (dC_K + dP_K) / S_e,
        "short_straddle":  -(dC_K + dP_K) / S_e,
        "short_iron_fly":  (-(dC_K + dP_K) + (dC_U + dP_L)) / S_e,
        "bull_call_spread": (dC_K - dC_U) / S_e,
        "bear_put_spread":  (dP_K - dP_L) / S_e,
        "risk_reversal":    (dC_U - dP_L) / S_e,
        "atm_put_write":   -dP_K / S_e,
    })
    return out
strategy_pnl.__wrapped__ = _base_pnl
print(f"{len(STRATEGIES)} strategies:", ", ".join(STRATEGIES))

17 strategies: stock, long_call, covered_call, protective_put, collar, cash_secured_put, short_strangle, drop_call, drop_put, long_put, long_straddle, short_straddle, short_iron_fly, bull_call_spread, bear_put_spread, risk_reversal, atm_put_write


## 7 · Scoreboard, placebo and plotting helpers

In [18]:
def bootstrap_ci(x, n_boot=2000, seed=0):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    if len(x) < 5:
        return (np.nan, np.nan)
    means = np.random.default_rng(seed).choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]).tolist())


def slice_results(res, bucket, entry, otm):
    if res is None or res.empty:
        return pd.DataFrame(columns=["horizon"] + STRATEGIES)
    return res[(res.bucket == bucket) & (res.entry == entry) & (res.otm == otm)]


def scoreboard(res, bucket=None, entry=None, otm=None, horizons=None, strategies=STRATEGIES):
    r = slice_results(res, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    horizons = horizons or [h for h in HORIZONS + ["exp"] if h in set(r.horizon)]
    rows = []
    for s in strategies:
        for h in horizons:
            x = r.loc[r.horizon == h, s].dropna() if s in r else pd.Series(dtype=float)
            lo, hi = bootstrap_ci(x)
            rows.append({"strategy": s, "horizon": h, "n": len(x), "mean": x.mean(), "ci_lo": lo, "ci_hi": hi,
                         "median": x.median(), "hit_rate": (x > 0).mean() if len(x) else np.nan})
    return pd.DataFrame(rows)


def difference_board(res_a, res_b, bucket=None, entry=None, otm=None, strategies=STRATEGIES, seed=1):
    a = slice_results(res_a, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    b = slice_results(res_b, bucket or BASELINE_BUCKET, entry or ENTRY, otm or OTM_PCT)
    rng, rows = np.random.default_rng(seed), []
    for s in strategies:
        for h in [h for h in HORIZONS + ["exp"] if h in set(a.horizon) & set(b.horizon)]:
            xa, xb = a.loc[a.horizon == h, s].dropna().to_numpy(), b.loc[b.horizon == h, s].dropna().to_numpy()
            row = {"strategy": s, "horizon": h, "n_a": len(xa), "n_b": len(xb)}
            if len(xa) >= 5 and len(xb) >= 5:
                d = rng.choice(xa, (2000, len(xa))).mean(1) - rng.choice(xb, (2000, len(xb))).mean(1)
                row.update(mean_a=xa.mean(), mean_b=xb.mean(), difference=xa.mean() - xb.mean(),
                           ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5))
            rows.append(row)
    return pd.DataFrame(rows)


def fmt_board(board, value="mean"):
    if board is None or board.empty or value not in board:
        return pd.DataFrame()
    def cell(row):
        v = row[value]
        if pd.isna(v):
            return ""
        star = "*" if pd.notna(row.get("ci_lo")) and (row["ci_lo"] > 0 or row["ci_hi"] < 0) else ""
        return f"{v * 100:+.2f}%{star}"
    wide = board.assign(cell=board.apply(cell, axis=1)).pivot(index="strategy", columns="horizon", values="cell")
    wide = wide.reindex([s for s in STRATEGIES if s in wide.index]).rename(index=STRATEGY_LABEL)
    wide.columns = [f"h={c}" if c != "exp" else "expiry" for c in wide.columns]
    return wide


def sample_placebo(ev, n, start, end, gap_days=PLACEBO_GAP_DAYS, seed=7):
    rng = np.random.default_rng(seed)
    sessions = CAL[(CAL >= pd.Timestamp(start)) & (CAL <= min(pd.Timestamp(end), LAST_SESSION))]
    by_ticker = ev.groupby("ticker")["filing_date"].apply(list)
    rows = []
    for t in rng.choice(ev["ticker"].to_numpy(), size=n, replace=True):
        for _ in range(25):
            d = sessions[rng.integers(len(sessions))]
            if all(abs((d - a).days) > gap_days for a in by_ticker.get(t, [])):
                rows.append({"ticker": t, "filing_date": d, "event_date": d, "t_0": d, "t_pre": session_before(d)})
                break
    return pd.DataFrame(rows)


SERIES = {"events": "#2a78d6", "placebo": "#eb6834", "oos": "#1baf7a", "in-sample": "#2a78d6", "holdout": "#8a4fd6"}


def plot_scoreboards(boards, title, strategies=STRATEGIES):
    boards = {k: v for k, v in boards.items() if v is not None and not v.empty}
    if not boards:
        return
    horizons = [h for h in HORIZONS + ["exp"] if any(h in set(b.horizon) for b in boards.values())]
    x = np.arange(len(horizons))
    cols = 3; rows = math.ceil(len(strategies) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(13, 3.2 * rows), sharex=True, squeeze=False)
    flat = axes.ravel()
    for ax, s in zip(flat, strategies):
        for i, (name, b) in enumerate(boards.items()):
            t = b[b.strategy == s].set_index("horizon").reindex(horizons)
            c = SERIES.get(name, "#52514e")
            if i == 0:
                ax.fill_between(x, t.ci_lo.to_numpy(float) * 100, t.ci_hi.to_numpy(float) * 100, color=c, alpha=0.12, linewidth=0)
            ax.plot(x, t["mean"].to_numpy(float) * 100, color=c, marker="o", markersize=4, linestyle="-" if i == 0 else "--",
                    label=f"{name} (n={int(np.nanmax(t.n)) if t.n.notna().any() else 0})")
        ax.axhline(0, color="#898781", linewidth=0.8)
        ax.set_xticks(x, [str(h) for h in horizons]); ax.set_title(STRATEGY_LABEL[s], loc="left", fontsize=10)
        ax.grid(axis="y", alpha=0.3)
    for ax in flat[len(strategies):]:
        ax.set_visible(False)
    flat[0].legend(frameon=False, fontsize=8)
    fig.suptitle(title, x=0.01, ha="left", fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.show()


def decay_table(r):
    rows = []
    for h in [h for h in HORIZONS + ["exp"] if h in set(r.horizon)]:
        x = r.loc[r.horizon == h, "ratio"].dropna()
        lo, hi = bootstrap_ci(x)
        rows.append({"horizon": h, "n": len(x), "mean_ratio": round(x.mean(), 2), "ci_lo": round(lo, 2), "ci_hi": round(hi, 2),
                     "median_ratio": round(x.median(), 2), "share_above_1": f"{(x > 1).mean():.0%}"})
    return pd.DataFrame(rows).set_index("horizon")

## 8 · `run_study`: the whole rule as one function of tags, start and end

In-sample, out-of-sample and the sealed window all go through this function, so the judges' run applies exactly your event definition, timing fix and filters.

In [19]:
def run_study(start, end, tags=None, universe=None, buckets=None, otm_pcts=None, with_placebo=False, label=None):
    tags = tags or EVENT_TAGS
    label = label or f"{'+'.join(tags)} {start}..{end}"
    ev = build_events(tags, start, end, universe or TOP_100)
    if MAX_EVENTS:
        ev = ev.head(MAX_EVENTS).copy()
    out = {"events": ev, "priced": [], "results": pd.DataFrame(), "board": None}
    if ev.empty:
        return out
    pr, drops = price_events(ev, buckets, otm_pcts, label=label)
    res = evaluate(pr, otm_pcts)
    out.update(priced=pr, dropped=drops, results=res, board=scoreboard(res) if not res.empty else None)
    if with_placebo:
        pev = sample_placebo(ev, N_PLACEBO, start, end)
        ppr, _ = price_events(pev, buckets, otm_pcts, label=f"{label} placebo")
        pres = evaluate(ppr, otm_pcts)
        out.update(placebo_results=pres, placebo_board=scoreboard(pres) if not pres.empty else None,
                   diff=difference_board(res, pres) if not pres.empty else None)
    return out

In [20]:
# ── Screen candidate pools (exploration only; log the pick in section 0 before the full run) ──
POOLS = {
    "A_resolution":  ["ceo_appointment", "cfo_appointment", "acquisition_completion", "settlement_agreement"],
    "B_gov_shock":   ["ceo_departure", "cfo_departure"],
    "C_bad_news":    ["material_charge_or_gain", "asset_impairment", "restructuring_plan", "regulatory_investigation"],
    "D_cap_return":  ["share_repurchase_program", "tender_offer"],
    "E_deal_news":   ["merger_agreement", "acquisition_agreement", "deal_termination"],
    "Z_control":     ["annual_meeting_results", "bylaw_amendment"],
}
SCREEN_H = [5, 21]
_b = {BASELINE_BUCKET: EXPIRY_BUCKETS[BASELINE_BUCKET]}
_o = [OTM_PCT]

screen = {}
for name, tags in POOLS.items():
    print(f"\n=== {name}: {tags}")
    screen[name] = run_study(STUDY_START, STUDY_END, tags=tags, buckets=_b, otm_pcts=_o, label=name)

# one shared placebo over all names that appear in any pool
all_ev = pd.concat([s["events"] for s in screen.values() if not s["events"].empty], ignore_index=True)
pev = sample_placebo(all_ev, 200, STUDY_START, STUDY_END)
ppr, _ = price_events(pev, _b, _o, label="screen placebo")
pres = evaluate(ppr, _o)

def _ratio(res, h):
    r = slice_results(res, BASELINE_BUCKET, "pre", OTM_PCT)
    x = r.loc[r.horizon == h, "ratio"].dropna()
    return f"{x.mean():.2f} (n={len(x)})" if len(x) else ""

rows = []
for name, s in screen.items():
    res = s["results"]
    row = {"pool": name, "events": len(s["events"])}
    if res.empty:
        rows.append(row); continue
    for h in SCREEN_H:
        row[f"decay h={h}"] = _ratio(res, h)
    d = difference_board(res, pres)
    for h in SCREEN_H:
        for _, r in d[d.horizon == h].iterrows():
            if pd.isna(r.get("difference")):
                continue
            star = "*" if (r.ci_lo > 0 or r.ci_hi < 0) else ""
            row[f"{STRATEGY_LABEL.get(r.strategy, r.strategy)} h={h}"] = f"{r.difference*100:+.2f}%{star}"
    rows.append(row)

placebo_row = {"pool": "PLACEBO", "events": len(pev), **{f"decay h={h}": _ratio(pres, h) for h in SCREEN_H}}
screen_tbl = pd.DataFrame(rows + [placebo_row]).set_index("pool")
screen_tbl.to_csv(OUT_DIR / "pool_screen.csv")
print("Event minus placebo, per $1 spot (3-6m, entry =", ENTRY, ").  * = 95% CI excludes 0")
display(screen_tbl.T)


=== A_resolution: ['ceo_appointment', 'cfo_appointment', 'acquisition_completion', 'settlement_agreement']
  acceptance times: 100% found, 40% after the close (moved a session)
97 events across 59 tickers for ['ceo_appointment', 'cfo_appointment', 'acquisition_completion', 'settlement_agreement'], 2024-01-01..2025-12-31
  A_resolution: 25/97 events, 44s
  A_resolution: 50/97 events, 86s
  A_resolution: 75/97 events, 132s
A_resolution: 97 events -> 89 priced (event, bucket) pairs; 8 drops

=== B_gov_shock: ['ceo_departure', 'cfo_departure']
  acceptance times: 100% found, 48% after the close (moved a session)
61 events across 41 tickers for ['ceo_departure', 'cfo_departure'], 2024-01-01..2025-12-31
  B_gov_shock: 25/61 events, 35s
  B_gov_shock: 50/61 events, 56s
B_gov_shock: 61 events -> 60 priced (event, bucket) pairs; 1 drops

=== C_bad_news: ['material_charge_or_gain', 'asset_impairment', 'restructuring_plan', 'regulatory_investigation']
  acceptance times: 100% found, 44% after th

pool,A_resolution,B_gov_shock,C_bad_news,D_cap_return,E_deal_news,Z_control,PLACEBO
events,97,61,54,49,78,193,200
decay h=5,1.03 (n=87),1.18 (n=59),1.33 (n=51),1.26 (n=40),0.92 (n=69),0.87 (n=168),1.01 (n=179)
decay h=21,0.98 (n=86),1.06 (n=57),0.96 (n=54),1.05 (n=40),0.84 (n=66),0.79 (n=167),0.92 (n=174)
Stock only (synthetic) h=5,-0.57%,-1.19%,+0.13%,-1.77%*,-0.63%,+0.32%,NaN
1 · Long call h=5,-0.24%,-0.46%,+0.23%,-0.97%*,-0.31%,+0.08%,NaN
2 · Covered call h=5,-0.35%,-0.84%,-0.02%,-1.27%*,-0.40%,+0.23%,NaN
3 · Protective put h=5,-0.35%,-0.75%,+0.27%,-1.23%*,-0.30%,+0.22%,NaN
4 · Collar h=5,-0.07%,-0.33%,+0.14%,-0.70%*,-0.16%,+0.14%,NaN
5 · Cash-secured put h=5,-0.29%,-0.52%*,-0.19%,-0.52%,-0.10%,+0.04%,NaN
Short strangle (benchmark) h=5,+0.03%,-0.07%,-0.21%,-0.01%,+0.08%,+0.02%,NaN


In [25]:
# ── CHECK: C_bad_news and B_gov_shock, done properly ──
CHECK = {"C_bad_news": POOLS["C_bad_news"], "B_gov_shock": POOLS["B_gov_shock"]}
FOCUS = ["stock", "cash_secured_put", "covered_call", "protective_put", "risk_reversal",
         "long_put", "long_straddle", "atm_put_write"]
FOCUS = [s for s in FOCUS if s in STRATEGIES]
_b = {BASELINE_BUCKET: EXPIRY_BUCKETS[BASELINE_BUCKET]}; _o = [OTM_PCT]

def _fmt_diff(d, hs=(1, 5, 21, 42)):
    d = d[d.strategy.isin(FOCUS) & d.horizon.isin(hs)].copy()
    d["cell"] = d.apply(lambda r: "" if pd.isna(r.get("difference")) else
                        f"{r.difference*100:+.2f}%{'*' if (r.ci_lo > 0 or r.ci_hi < 0) else ''}", axis=1)
    w = d.pivot(index="strategy", columns="horizon", values="cell").reindex(FOCUS)
    return w.rename(index=STRATEGY_LABEL)

check = {}
for name, tags in CHECK.items():
    ins = screen[name] if name in screen else run_study(STUDY_START, STUDY_END, tags=tags, buckets=_b, otm_pcts=_o)
    pev = sample_placebo(ins["events"], 150, STUDY_START, STUDY_END)        # same tickers as the pool
    ppr, _ = price_events(pev, _b, _o, label=f"{name} placebo")
    pres = evaluate(ppr, _o)
    oos = run_study(OOS_START, OOS_END, tags=tags, buckets=_b, otm_pcts=_o, label=f"{name} OOS")
    check[name] = dict(ins=ins, pres=pres, oos=oos)

    print(f"\n================ {name}  (in-sample n={len(ins['events'])}, tickers={ins['events'].ticker.nunique()}, "
          f"OOS n={len(oos['events'])})")
    for entry in ("pre", "post"):
        print(f"\nIn-sample, event − own-ticker placebo, entry={entry}  (* = 95% CI excludes 0)")
        display(_fmt_diff(difference_board(ins["results"], pres, entry=entry)))

    r = slice_results(ins["results"], BASELINE_BUCKET, ENTRY, OTM_PCT)
    r21 = r[r.horizon == 21]
    trim = {STRATEGY_LABEL.get(s, s): f"{r21[s].mean()*100:+.2f}% → {r21[s].sort_values().iloc[3:-3].mean()*100:+.2f}%"
            for s in FOCUS if r21[s].notna().sum() > 10}
    print("h=21 mean → mean without top/bottom 3 events:"); display(pd.Series(trim).to_frame("raw → trimmed"))
    if "ticker" in r21:
        print("Most influential tickers at h=21 (stock P&L sum):")
        display((r21.groupby("ticker")["stock"].sum() * 100).sort_values().iloc[[0, 1, 2, -3, -2, -1]].round(1))

    if not oos["results"].empty:
        o = slice_results(oos["results"], BASELINE_BUCKET, ENTRY, OTM_PCT)
        cmp = {}
        for s in FOCUS:
            for h in (5, 21):
                a = r.loc[r.horizon == h, s].mean(); b = o.loc[o.horizon == h, s].mean()
                cmp[(STRATEGY_LABEL.get(s, s), f"h={h}")] = f"{a*100:+.2f}% → {b*100:+.2f}% {'✓' if np.sign(a) == np.sign(b) else '✗'}"
        print("In-sample → OOS raw mean (no placebo), same sign?")
        display(pd.Series(cmp).unstack())

  C_bad_news placebo: 25/150 events, 0s
  C_bad_news placebo: 50/150 events, 1s
  C_bad_news placebo: 75/150 events, 1s
  C_bad_news placebo: 100/150 events, 1s
  C_bad_news placebo: 125/150 events, 1s
  C_bad_news placebo: 150/150 events, 1s
C_bad_news placebo: 150 events -> 143 priced (event, bucket) pairs; 7 drops
  acceptance times: 100% found, 64% after the close (moved a session)
14 events across 13 tickers for ['material_charge_or_gain', 'asset_impairment', 'restructuring_plan', 'regulatory_investigation'], 2026-01-01..2026-08-31
C_bad_news OOS: 14 events -> 11 priced (event, bucket) pairs; 3 drops

================ C_bad_news  (in-sample n=54, tickers=31, OOS n=14)

In-sample, event − own-ticker placebo, entry=pre  (* = 95% CI excludes 0)


horizon,1,5,21,42
strategy,,,,
Stock only (synthetic),+0.29%,+0.04%,+2.03%,+2.57%
5 · Cash-secured put,+0.11%,-0.05%,+0.55%,+0.57%
2 · Covered call,+0.16%,-0.05%,+1.15%,+0.93%
3 · Protective put,-0.06%,+0.11%,+1.28%,+1.99%
Ext · Risk reversal (+C_U −P_L),+0.09%,+0.05%,+1.29%,+1.93%
Ext · Long put (ATM),-0.11%,+0.04%,-0.87%*,-0.76%
Ext · Long straddle (ATM),+0.05%,+0.23%,+0.28%,+1.15%
Ext · ATM put-write,+0.11%,-0.04%,+0.87%*,+0.76%



In-sample, event − own-ticker placebo, entry=post  (* = 95% CI excludes 0)


horizon,1,5,21,42
strategy,,,,
Stock only (synthetic),+0.28%,+0.20%,+1.99%,+2.58%
5 · Cash-secured put,+0.05%,-0.15%,+0.46%,+0.49%
2 · Covered call,+0.22%,+0.03%,+1.20%*,+0.90%
3 · Protective put,+0.23%,+0.35%,+1.49%,+2.15%
Ext · Risk reversal (+C_U −P_L),+0.12%,-0.06%,+1.24%,+2.02%
Ext · Long put (ATM),-0.09%,+0.10%,-0.85%,-0.62%
Ext · Long straddle (ATM),+0.10%,+0.39%,+0.30%,+1.33%
Ext · ATM put-write,+0.09%,-0.10%,+0.85%,+0.62%


h=21 mean → mean without top/bottom 3 events:


...

,raw → trimmed
Stock only (synthetic),+1.36% → +2.24%
5 · Cash-secured put,+0.01% → +0.35%
2 · Covered call,+0.49% → +1.09%
3 · Protective put,+0.93% → +1.51%
Ext · Risk reversal (+C_U −P_L),+0.55% → +1.18%
Ext · Long put (ATM),-0.18% → -0.04%
Ext · Long straddle (ATM),+0.87% → +1.17%
Ext · ATM put-write,+0.18% → +0.30%


Most influential tickers at h=21 (stock P&L sum):


ticker
BA     -18.0
ORCL   -15.6
TGT    -10.3
C       19.3
NFLX    21.3
INTC    43.1
Name: stock, dtype: float64

In-sample → OOS raw mean (no placebo), same sign?


,h=21,h=5
2 · Covered call,+0.49% → +1.93% ✓,-0.49% → +0.18% ✗
3 · Protective put,+0.93% → +3.67% ✓,-0.43% → +0.58% ✗
5 · Cash-secured put,+0.01% → +0.65% ✓,-0.34% → +0.06% ✗
Ext · ATM put-write,+0.18% → +1.26% ✓,-0.53% → +0.13% ✗
Ext · Long put (ATM),-0.18% → -1.26% ✓,+0.53% → -0.13% ✗
Ext · Long straddle (ATM),+0.87% → +1.52% ✓,+0.24% → +0.29% ✓
Ext · Risk reversal (+C_U −P_L),+0.55% → +2.81% ✓,-0.63% → +0.52% ✗
Stock only (synthetic),+1.36% → +4.41% ✓,-0.74% → +0.64% ✗


In [26]:
# ── OOS with placebo + pooled B∪C test ──
FOC = [s for s in ["stock", "protective_put", "cash_secured_put", "covered_call",
                   "long_put", "long_straddle", "risk_reversal"] if s in STRATEGIES]
HS  = (1, 3, 5, 10, 21)

def _cells(d):
    d = d[d.strategy.isin(FOC) & d.horizon.isin(HS)].copy()
    d["cell"] = d.apply(lambda r: "" if pd.isna(r.get("difference")) else
                        f"{r.difference*100:+.2f}%{'*' if (r.ci_lo > 0 or r.ci_hi < 0) else ''}", axis=1)
    return d.pivot(index="strategy", columns="horizon", values="cell").reindex(FOC).rename(index=STRATEGY_LABEL)

for name, c in check.items():
    if "oos_pres" not in c and not c["oos"]["events"].empty:
        pev = sample_placebo(c["oos"]["events"], 100, OOS_START, OOS_END)
        ppr, _ = price_events(pev, _b, _o, label=f"{name} OOS placebo")
        c["oos_pres"] = evaluate(ppr, _o)

# pooled B ∪ C (dedupe events that carry both)
def _pool(key):
    return pd.concat([check[n][key]["results"] if isinstance(check[n][key], dict) else check[n][key]
                      for n in ("B_gov_shock", "C_bad_news")], ignore_index=True)
pooled = {"ins": _pool("ins"), "pres": _pool("pres"), "oos": _pool("oos"), "oos_pres": _pool("oos_pres")}
if "ticker" in pooled["ins"] and "t_0" in pooled["ins"]:
    for k in ("ins", "oos"):
        pooled[k] = pooled[k].drop_duplicates(["ticker", "t_0", "bucket", "entry", "otm", "horizon"])

for name, c in [("B_gov_shock", check["B_gov_shock"]), ("C_bad_news", check["C_bad_news"]),
                ("POOLED B∪C", {"ins": {"results": pooled["ins"]}, "pres": pooled["pres"],
                                "oos": {"results": pooled["oos"]}, "oos_pres": pooled["oos_pres"]})]:
    print(f"\n================ {name}")
    for entry in ("pre", "post"):
        a = _cells(difference_board(c["ins"]["results"], c["pres"], entry=entry))
        b = _cells(difference_board(c["oos"]["results"], c["oos_pres"], entry=entry))
        print(f"entry={entry}: event − own-ticker placebo   [in-sample | OOS]")
        display(pd.concat({"in-sample": a, "OOS": b}, axis=1))

# robustness of the h=5 effect in B: trimmed mean + hit rate
r5 = slice_results(check["B_gov_shock"]["ins"]["results"], BASELINE_BUCKET, "post", OTM_PCT)
r5 = r5[r5.horizon == 5]
print("\nB h=5, post entry: raw → trimmed(3) mean, share of events with stock < 0")
display(pd.DataFrame({s: [f"{r5[s].mean()*100:+.2f}%", f"{r5[s].sort_values().iloc[3:-3].mean()*100:+.2f}%"]
                      for s in FOC}, index=["raw", "trimmed"]).rename(columns=STRATEGY_LABEL))
print(f"share stock<0 at h=5: {(r5.stock < 0).mean():.0%}  (n={len(r5)})")

  C_bad_news OOS placebo: 25/100 events, 15s
  C_bad_news OOS placebo: 50/100 events, 29s
  C_bad_news OOS placebo: 75/100 events, 43s
  C_bad_news OOS placebo: 100/100 events, 56s
C_bad_news OOS placebo: 100 events -> 86 priced (event, bucket) pairs; 14 drops
  B_gov_shock OOS placebo: 25/100 events, 17s
  B_gov_shock OOS placebo: 50/100 events, 34s
  B_gov_shock OOS placebo: 75/100 events, 51s
  B_gov_shock OOS placebo: 100/100 events, 66s
B_gov_shock OOS placebo: 100 events -> 91 priced (event, bucket) pairs; 9 drops

================ B_gov_shock
entry=pre: event − own-ticker placebo   [in-sample | OOS]


in-sample                                   \
horizon                                 1       3        5      10      21   
strategy                                                                     
Stock only (synthetic)             -0.50%  -0.67%   -1.59%  -0.71%  +0.38%   
3 · Protective put                 -0.36%  -0.50%  -1.19%*  -0.45%  +0.26%   
5 · Cash-secured put               -0.13%  -0.15%   -0.48%  -0.39%  -0.26%   
2 · Covered call                   -0.40%  -0.43%   -0.98%  -0.60%  -0.38%   
Ext · Long put (ATM)               +0.31%  +0.40%  +0.79%*  +0.55%  +0.22%   
Ext · Long straddle (ATM)          +0.13%  +0.16%   -0.01%  +0.34%  +0.93%   
Ext · Risk reversal (+C_U −P_L)    -0.26%  -0.44%  -1.17%*  -0.52%  +0.18%   

                                     OOS                                   
horizon                                1        3       5      10      21  
strategy                                                                   
Stock only (synthetic)            -1.29%   -1.99%  -1.76%  -1.45%  +2.56%  
3 · Protective put                -0.93%   -1.33%  -1.27%  -1.11%  +1.68%  
5 · Cash-secured put              -0.32%   -0.53%  -0.45%  -0.36%  +0.17%  
2 · Covered call                  -0.45%   -0.79%  -0.98%  -1.02%  +0.67%  
Ext · Long put (ATM)              +0.48%  +0.74%*  +0.87%  +0.88%  -0.55%  
Ext · Long straddle (ATM)         -0.32%   -0.50%  -0.02%  +0.13%  +1.41%  
Ext · Risk reversal (+C_U −P_L)  -1.08%*  -1.50%*  -1.19%  -0.32%  +1.46%

entry=post: event − own-ticker placebo   [in-sample | OOS]


in-sample                                   \
horizon                                 1       3        5      10      21   
strategy                                                                     
Stock only (synthetic)            -0.80%*  -0.99%  -1.90%*  -1.02%  +0.07%   
3 · Protective put                -0.47%*  -0.63%  -1.33%*  -0.59%  +0.12%   
5 · Cash-secured put               -0.28%  -0.31%  -0.66%*  -0.57%  -0.39%   
2 · Covered call                  -0.54%*  -0.57%  -1.14%*  -0.80%  -0.53%   
Ext · Long put (ATM)              +0.47%*  +0.56%  +0.94%*  +0.70%  +0.38%   
Ext · Long straddle (ATM)          +0.14%  +0.14%   -0.01%  +0.37%  +0.96%   
Ext · Risk reversal (+C_U −P_L)   -0.54%*  -0.73%  -1.48%*  -0.80%  -0.12%   

                                    OOS                                  
horizon                               1       3       5      10      21  
strategy                                                                 
Stock only (synthetic)           +0.69%  -0.04%  +0.11%  +0.47%  +4.66%  
3 · Protective put               +0.58%  +0.17%  +0.15%  +0.34%  +3.30%  
5 · Cash-secured put             +0.11%  -0.09%  -0.02%  +0.09%  +0.64%  
2 · Covered call                 +0.51%  +0.18%  -0.04%  -0.12%  +1.60%  
Ext · Long put (ATM)             -0.40%  -0.14%  +0.04%  +0.03%  -1.50%  
Ext · Long straddle (ATM)        -0.12%  -0.33%  +0.18%  +0.36%  +1.52%  
Ext · Risk reversal (+C_U −P_L)  +0.28%  -0.12%  +0.21%  +1.01%  +2.88%


================ C_bad_news
entry=pre: event − own-ticker placebo   [in-sample | OOS]


...

entry=post: event − own-ticker placebo   [in-sample | OOS]


in-sample                                   \
horizon                                 1       3        5      10      21   
strategy                                                                     
Stock only (synthetic)             -0.28%  -0.23%   -0.89%  -0.74%  +1.00%   
3 · Protective put                 -0.13%  -0.08%   -0.51%  -0.40%  +0.79%   
5 · Cash-secured put               -0.12%  -0.13%  -0.41%*  -0.42%  +0.02%   
2 · Covered call                   -0.18%  -0.09%   -0.58%  -0.53%  +0.31%   
Ext · Long put (ATM)               +0.20%  +0.17%  +0.54%*  +0.53%  -0.21%   
Ext · Long straddle (ATM)          +0.12%  +0.12%   +0.19%  +0.33%  +0.63%   
Ext · Risk reversal (+C_U −P_L)    -0.22%  -0.28%  -0.78%*  -0.63%  +0.54%   

                                     OOS                                  
horizon                                1       3       5      10      21  
strategy                                                                  
Stock only (synthetic)            +0.70%  +0.50%  +0.40%  +0.48%  +0.96%  
3 · Protective put                +0.54%  +0.58%  +0.35%  +0.64%  +1.24%  
5 · Cash-secured put              +0.15%  -0.03%  +0.05%  -0.15%  -0.11%  
2 · Covered call                 +0.44%*  +0.48%  -0.05%  -0.39%  -0.51%  
Ext · Long put (ATM)              -0.32%  -0.20%  +0.07%  +0.24%  +0.23%  
Ext · Long straddle (ATM)         +0.04%  +0.09%  +0.54%  +0.81%  +1.69%  
Ext · Risk reversal (+C_U −P_L)   +0.40%  +0.05%  +0.55%  +0.92%  +1.01%


B h=5, post entry: raw → trimmed(3) mean, share of events with stock < 0


,Stock only (synthetic),3 · Protective put,5 · Cash-secured put,2 · Covered call,Ext · Long put (ATM),Ext · Long straddle (ATM),Ext · Risk reversal (+C_U −P_L)
raw,-0.74%,-0.43%,-0.34%,-0.49%,+0.53%,+0.24%,-0.63%
trimmed,-0.39%,-0.18%,-0.09%,-0.24%,+0.46%,+0.21%,-0.26%


share stock<0 at h=5: 52%  (n=60)


In [ ]:
# ── EXPLORE 1: broad sweep, tags × strategies × horizons ──
MIN_EVENTS  = 25
SWEEP_H     = [1, 5, 21]
EXCLUDE     = {"quarterly_earnings", "preliminary_results"}   # earnings; keep as-is or remove from the set
_b = {BASELINE_BUCKET: EXPIRY_BUCKETS[BASELINE_BUCKET]}
_o = [OTM_PCT]

tc = pd.read_csv(OUT_DIR / "tag_counts_in_sample.csv")
groups = {t: [t] for t in tc.loc[tc.events_in_top100 >= MIN_EVENTS, "tag"] if t not in EXCLUDE}
groups.update({f"POOL_{k}": v for k, v in (POOLS if "POOLS" in globals() else {}).items()})

sweep = {}
for i, (name, tags) in enumerate(groups.items(), 1):
    print(f"[{i}/{len(groups)}] {name}")
    try:
        sweep[name] = run_study(STUDY_START, STUDY_END, tags=tags, buckets=_b, otm_pcts=_o, label=name)
    except Exception as err:
        print("   failed:", err)

_all = pd.concat([s["events"] for s in sweep.values() if not s["events"].empty], ignore_index=True)
_pev = sample_placebo(_all.drop_duplicates(["ticker", "filing_date"]), 300, STUDY_START, STUDY_END)
_ppr, _ = price_events(_pev, _b, _o, label="sweep placebo")
sweep_placebo = evaluate(_ppr, _o)

def _decay(res, h):
    r = slice_results(res, BASELINE_BUCKET, "pre", OTM_PCT)
    x = r.loc[r.horizon == h, "ratio"].dropna()
    return x.mean() if len(x) >= 5 else np.nan

diff_rows, decay_rows = [], []
for name, s in sweep.items():
    if s["results"].empty:
        continue
    d = difference_board(s["results"], sweep_placebo)
    d = d[d.horizon.isin(SWEEP_H)].assign(group=name)
    diff_rows.append(d)
    decay_rows.append({"group": name, "n": len(s["events"]),
                       **{f"h={h}": _decay(s["results"], h) for h in SWEEP_H}})
decay_rows.append({"group": "PLACEBO", "n": len(_pev),
                   **{f"h={h}": _decay(sweep_placebo, h) for h in SWEEP_H}})

sweep_diff = pd.concat(diff_rows, ignore_index=True)
sweep_diff["sig"] = (sweep_diff.ci_lo > 0) | (sweep_diff.ci_hi < 0)
sweep_decay = pd.DataFrame(decay_rows).set_index("group").sort_values("h=5")
sweep_diff.to_csv(OUT_DIR / "sweep_diff.csv", index=False)
sweep_decay.to_csv(OUT_DIR / "sweep_decay.csv")

print("|realized| / implied (pre-event chain). >1 = market under-priced the move, <1 = over-priced")
display(sweep_decay.style.format("{:.2f}", subset=[f"h={h}" for h in SWEEP_H])
        .background_gradient(cmap="RdBu_r", vmin=0.6, vmax=1.4, subset=[f"h={h}" for h in SWEEP_H]))

fig, axes = plt.subplots(1, len(SWEEP_H), figsize=(6 * len(SWEEP_H), 0.38 * len(sweep) + 2), sharey=True)
for ax, h in zip(np.atleast_1d(axes), SWEEP_H):
    m = sweep_diff[sweep_diff.horizon == h].pivot(index="group", columns="strategy", values="difference") * 100
    sig = sweep_diff[sweep_diff.horizon == h].pivot(index="group", columns="strategy", values="sig")
    m = m[[c for c in STRATEGIES if c in m]]; sig = sig.reindex_like(m)
    lim = np.nanpercentile(np.abs(m.values), 95)
    im = ax.imshow(m.values, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
    for (r, c), v in np.ndenumerate(m.values):
        if not np.isnan(v):
            ax.text(c, r, f"{v:+.1f}{'*' if sig.values[r, c] else ''}", ha="center", va="center", fontsize=7)
    ax.set_xticks(range(m.shape[1])); ax.set_xticklabels([STRATEGY_LABEL.get(c, c) for c in m.columns], rotation=70, ha="right", fontsize=8)
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index, fontsize=8)
    ax.set_title(f"event − placebo, % of spot, h={h}")
plt.tight_layout(); plt.show()

print("Top 15 significant cells:")
display(sweep_diff[sweep_diff.sig].assign(abs_d=lambda d: d.difference.abs())
        .sort_values("abs_d", ascending=False).head(15)
        [["group", "strategy", "horizon", "n_a", "difference", "ci_lo", "ci_hi"]])

In [ ]:
# ── EXPLORE 2: drill into one tag or pool ──
DRILL     = "ceo_departure"                       # any key of `sweep`
STRATS    = ["stock", "protective_put", "long_straddle", "short_iron_fly", "cash_secured_put"]
DRILL_H   = 21

s   = sweep[DRILL]
res = slice_results(s["results"], BASELINE_BUCKET, ENTRY, OTM_PCT)
pl  = slice_results(sweep_placebo, BASELINE_BUCKET, ENTRY, OTM_PCT)
hs  = [h for h in HORIZONS if h in set(res.horizon)]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for st in STRATS:
    ev_m = [res.loc[res.horizon == h, st].mean() * 100 for h in hs]
    pl_m = [pl.loc[pl.horizon == h, st].mean() * 100 for h in hs]
    l, = ax[0].plot(hs, ev_m, marker="o", label=STRATEGY_LABEL.get(st, st))
    ax[0].plot(hs, pl_m, ls="--", color=l.get_color(), alpha=.5)
ax[0].axhline(0, c="k", lw=.5); ax[0].set_xscale("log"); ax[0].set_xticks(hs); ax[0].set_xticklabels(hs)
ax[0].set_title(f"{DRILL}: mean P&L (solid) vs placebo (dashed)"); ax[0].legend(fontsize=7)

pre_ev = slice_results(s["results"], BASELINE_BUCKET, "pre", OTM_PCT)
pre_pl = slice_results(sweep_placebo, BASELINE_BUCKET, "pre", OTM_PCT)
for lab, r, c in (("events", pre_ev, "C3"), ("placebo", pre_pl, "C7")):
    m  = [r.loc[r.horizon == h, "ratio"].mean() for h in hs]
    md = [r.loc[r.horizon == h, "ratio"].median() for h in hs]
    ax[1].plot(hs, m, marker="o", c=c, label=f"{lab} mean"); ax[1].plot(hs, md, ls=":", c=c, label=f"{lab} median")
ax[1].axhline(1, c="k", lw=.5); ax[1].set_xscale("log"); ax[1].set_xticks(hs); ax[1].set_xticklabels(hs)
ax[1].set_title("|realized| / implied"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

per_ev = (res[res.horizon == DRILL_H][["ticker", "t_0"] + STRATS] if "ticker" in res else res[res.horizon == DRILL_H][STRATS])
per_ev = per_ev.copy(); per_ev[STRATS] = per_ev[STRATS] * 100
print(f"Per-event P&L at h={DRILL_H}, % of spot (sorted by stock move). Check whether 1-2 events drive the mean.")
display(per_ev.sort_values("stock").style.format("{:+.1f}", subset=STRATS)
        .background_gradient(cmap="RdBu", vmin=-15, vmax=15, subset=STRATS))
print("Mean without the top & bottom 2 events:",
      {st: f"{per_ev[st].sort_values().iloc[2:-2].mean():+.2f}%" for st in STRATS})
with pd.option_context("display.max_colwidth", 140):
    display(s["events"][["ticker", "filing_date", "tags", "supporting_text"]].head(10))

## 9 · In-sample: scoreboard, placebo gap and decay

Read the **event-minus-placebo** table, not the raw scoreboard. Short-option strategies earn the variance risk premium on ordinary days too.

In [ ]:
study = run_study(STUDY_START, STUDY_END, with_placebo=RUN_PLACEBO, label="in-sample")
results, board = study["results"], study["board"]
assert board is not None, "No priced in-sample events: choose more or different tags."
print(f"In-sample · {EVENT_TAGS} · {BASELINE_BUCKET} · entry={ENTRY} · OTM {OTM_PCT:.0%} · n={int(board.n.max())}. * = 95% CI excludes zero")
display(fmt_board(board))
display(study["events"].groupby("tags").size().rename("events").to_frame())

if RUN_PLACEBO and study.get("diff") is not None:
    plot_scoreboards({"events": board, "placebo": study["placebo_board"]}, "Events vs ordinary days for the same names")
    print("Events minus ordinary days (the finding). * = 95% CI on the difference excludes zero")
    display(fmt_board(study["diff"], value="difference"))

pre = slice_results(results, BASELINE_BUCKET, "pre", OTM_PCT)
tables = {"events": decay_table(pre)}
if RUN_PLACEBO and study.get("placebo_results") is not None:
    tables["placebo"] = decay_table(slice_results(study["placebo_results"], BASELINE_BUCKET, "pre", OTM_PCT))
print("|realized| / implied move from the pre-event chain (near 1 = priced right; below 1 = overpriced)")
display(pd.concat(tables, names=["group", "horizon"]))

## 10 · Out-of-sample

Same pipeline on dates nothing above touched. With a dozen or two events this checks the **sign**, not the size.

In [ ]:
oos = run_study(OOS_START, OOS_END, label="out-of-sample")
oos_board = oos["board"]
if oos_board is not None:
    print(f"Out-of-sample · n={int(oos_board.n.max())}")
    display(fmt_board(oos_board))
    comp = board.merge(oos_board, on=["strategy", "horizon"], suffixes=("_in", "_oos"))
    comp = comp[comp.horizon.isin([5, 21, 42, "exp"])]
    comp["cell"] = [f"{a * 100:+.2f}% → {b * 100:+.2f}% {'✓' if np.sign(a) == np.sign(b) else '✗'}"
                    for a, b in zip(comp.mean_in, comp.mean_oos)]
    w = comp.pivot(index="strategy", columns="horizon", values="cell").reindex(STRATEGIES).rename(index=STRATEGY_LABEL)
    print("In-sample → out-of-sample mean; ✓ = same sign"); display(w)
    plot_scoreboards({"in-sample": board, "oos": oos_board}, "In-sample vs out-of-sample")
else:
    print("No out-of-sample events for these tags.")

## 11 · Parameter sensitivity

A finding that holds at only one setting is not a finding. Read across each row: does the sign hold, and does the star survive?

In [ ]:
grid_rows = []
for b in EXPIRY_BUCKETS:
    for e in ("pre", "post"):
        for o in OTM_GRID:
            for _, r in scoreboard(results, bucket=b, entry=e, otm=o, horizons=[SENS_HORIZON]).iterrows():
                grid_rows.append({"bucket": b, "entry": e, "otm": o, "strategy": r.strategy, "mean": r["mean"], "ci_lo": r.ci_lo, "ci_hi": r.ci_hi})
grid = pd.DataFrame(grid_rows)
grid["cell"] = [f"{m * 100:+.2f}%{'*' if (lo > 0 or hi < 0) else ''}" if pd.notna(m) else "" for m, lo, hi in zip(grid["mean"], grid.ci_lo, grid.ci_hi)]


def sens_table(sub, columns):
    return (sub[sub.strategy != "stock"].pivot_table(index="strategy", columns=columns, values="cell", aggfunc="first")
              .reindex(STRATEGIES[1:]).rename(index=STRATEGY_LABEL))


print(f"OTM distance × entry, {BASELINE_BUCKET} bucket, h={SENS_HORIZON}")
display(sens_table(grid[grid.bucket == BASELINE_BUCKET], ["entry", "otm"]))
print(f"Expiry bucket × entry, OTM {OTM_PCT:.0%}, h={SENS_HORIZON}")
display(sens_table(grid[grid.otm == OTM_PCT], ["entry", "bucket"]))

## 12 · Trade specification (registered strategy)

Costs charge `COST_HAIRCUT` of each traded premium on entry and exit. Capacity uses each leg's volume on the entry session.

In [ ]:
legs = [f"{l}{OTM_PCT}" if l in ("C_U", "P_L") else l for l in LEGS_USED[HEADLINE_STRATEGY]]
hz = slice_results(results, BASELINE_BUCKET, ENTRY, OTM_PCT)
rows = hz[hz.horizon == SENS_HORIZON].copy()
if legs and not rows.empty:
    key = lambda pe: (pe.ticker, pe.event_date)
    base_priced = {key(pe): pe for pe in study["priced"] if pe.bucket == BASELINE_BUCKET}
    e_day = lambda pe: pe.t_0 if ENTRY == "post" else pe.t_pre
    rows["premium"] = [sum(abs(base_priced[(t, d)].marks(e_day(base_priced[(t, d)]))[l]) for l in legs) / s
                       for t, d, s in zip(rows.ticker, rows.event_date, rows.S_entry)]
    rows["cost"] = rows["premium"] * COST_HAIRCUT * 2
    rows["net"] = rows[HEADLINE_STRATEGY] - rows["cost"]
    rows["leg_volume"] = [sum(base_priced[(t, d)].legs[l].volume_on(e_day(base_priced[(t, d)])) for l in legs)
                          for t, d in zip(rows.ticker, rows.event_date)]
    display(pd.Series({
        "strategy": STRATEGY_LABEL[HEADLINE_STRATEGY], "events": len(rows),
        f"gross mean at h={SENS_HORIZON}": f"{rows[HEADLINE_STRATEGY].mean() * 100:+.2f}%",
        "mean round-trip cost": f"{rows['cost'].mean() * 100:.2f}%",
        f"net mean at h={SENS_HORIZON}": f"{rows['net'].mean() * 100:+.2f}%",
        "share net positive": f"{(rows['net'] > 0).mean():.0%}",
        "worst single event (net)": f"{rows['net'].min() * 100:+.2f}%",
        "median contracts traded in the legs at entry": f"{rows['leg_volume'].median():,.0f}",
    }).to_frame("value"))
else:
    print("No option legs for the registered strategy (stock) or no rows at this horizon.")

## 13 · Variant log and saved outputs

Each run appends one line to `outputs/variants_log.csv`. Report the total count in the write-up.

In [ ]:
def _get(df, s, h, col):
    if df is None or df.empty:
        return np.nan
    r = df[(df.strategy == s) & (df.horizon == h)]
    return float(r[col].iloc[0]) if len(r) and col in r and pd.notna(r[col].iloc[0]) else np.nan


log_row = {
    "run_at": datetime.now().isoformat(timespec="seconds"), "tags": "+".join(EVENT_TAGS), "strategy": HEADLINE_STRATEGY,
    "bucket": BASELINE_BUCKET, "otm": OTM_PCT, "entry": ENTRY, "earnings_filter_days": EXCLUDE_EARNINGS_DAYS,
    "acceptance_fix": FETCH_ACCEPTANCE_TIMES, "n_in": int(board.n.max()),
    f"mean_in_h{SENS_HORIZON}": _get(board, HEADLINE_STRATEGY, SENS_HORIZON, "mean"),
    f"diff_vs_placebo_h{SENS_HORIZON}": _get(study.get("diff"), HEADLINE_STRATEGY, SENS_HORIZON, "difference"),
    f"diff_ci_lo": _get(study.get("diff"), HEADLINE_STRATEGY, SENS_HORIZON, "ci_lo"),
    f"diff_ci_hi": _get(study.get("diff"), HEADLINE_STRATEGY, SENS_HORIZON, "ci_hi"),
    "n_oos": int(oos_board.n.max()) if oos_board is not None else 0,
    f"mean_oos_h{SENS_HORIZON}": _get(oos_board, HEADLINE_STRATEGY, SENS_HORIZON, "mean"),
    "note": VARIANT_NOTE,
}
log_path = OUT_DIR / "variants_log.csv"
pd.DataFrame([log_row]).to_csv(log_path, mode="a", header=not log_path.exists(), index=False)
variants = pd.read_csv(log_path)
print(f"{len(variants)} variants logged so far")
display(variants.tail(10))

tagname = "+".join(EVENT_TAGS)
board.to_csv(OUT_DIR / f"board_in_{tagname}.csv", index=False)
if study.get("diff") is not None:
    study["diff"].to_csv(OUT_DIR / f"diff_vs_placebo_{tagname}.csv", index=False)
if oos_board is not None:
    oos_board.to_csv(OUT_DIR / f"board_oos_{tagname}.csv", index=False)

## Sealed window · judges only

Judges set `RUN_HOLDOUT = True` and the holdout dates in section 2, then run the notebook. This cell runs the same `run_study`, so the event definition, timing fix and filters all carry over.

In [ ]:
if RUN_HOLDOUT:
    holdout = run_study(HOLDOUT_START, HOLDOUT_END, label="holdout")
    if holdout["board"] is not None:
        print(f"Sealed window {HOLDOUT_START}..{HOLDOUT_END}: n={int(holdout['board'].n.max())}")
        display(fmt_board(holdout["board"]))
        plot_scoreboards({"in-sample": board, "holdout": holdout["board"]}, "In-sample vs sealed window")
    else:
        print("No events in the sealed window for these tags.")
else:
    print(f"Sealed window not run. Judges: set RUN_HOLDOUT = True.")

## Write-up (at most two pages)

**1 · Hypothesis.** Copy from section 0 unchanged, with the commit date.

**2 · Method.** Tags combined; universe; event timing (filing date, acceptance-time correction, share moved a session); earnings filter; expiry bucket and strikes; entry and exit rule; placebo design; costs.

**3 · Results.** Event-minus-placebo at every fixed horizon with 95% intervals, in-sample and out-of-sample. The decay of the edge across horizons. Sensitivity across OTM, bucket and entry. The total number of variants tried (from the log).

**4 · What would make it break.** Predicted fragility for the sealed window: regime (calm vs volatile year), small samples, specific large events driving the mean, earnings overlap, survivorship of the static universe.

**5 · How to trade it.** Instrument, entry timing respecting the filing lag, exit horizon justified by the decay, costs as a share of gross edge, capacity from leg volume, worst single event and sizing.

**Data sources.** Massive Filings & Disclosures (taxonomy version: ___), Massive options contracts and aggregates, SEC EDGAR acceptance timestamps.

## Known limitations (carry into the write-up)

- **Static universe:** today's top 100 applied backwards (survivorship bias).
- **Inferred stock price:** spot comes from put-call parity with a flat rate and no dividends; `abs_spot_gap` was typically under 0.5% in the starter.
- **Synthetic stock:** collects no dividends and ignores early assignment.
- **Last-trade marks:** no bid-ask spread except the flat haircut; deep in-the-money legs can be stale.
- **Short strangle:** used only as a benchmark; the naked short call is not a library strategy and carries margin and assignment risk.
- **Small samples:** wide intervals by construction. Combining tags trades sample size against a cleaner event definition.
- **Earnings filter:** relies on earnings-related 8-K tags, whose coverage may be thin, so treat it as a lower bound.
- **Filing lag:** an 8-K can follow the event by up to four business days, so the news may already be priced at `t_0`.